# 01 · Exploring the dataset

Design variables (features) and functional attributes (targets) of midsize sedans.
Data: `data/midsize_sedans.csv`, 31 midsize sedans (model years around 2000) from
de Weck's product-platform study. Set the `DATA_PATH` environment variable to use
another CSV with the same columns.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from smalldata import FEATURES, TARGETS, load_dataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Set the DATA_PATH environment variable to run on another CSV with the same columns
DATA_PATH = Path(os.environ.get("DATA_PATH", ROOT / "data" / "midsize_sedans.csv"))
ds = load_dataset(DATA_PATH)
print(f"{ds.n_samples} vehicles, {len(FEATURES)} design variables, {len(TARGETS)} functional attributes")

## Summary statistics and missing values

In [ ]:
summary = pd.concat([ds.X, ds.Y], axis=1).describe().T
summary["missing"] = pd.concat([ds.X, ds.Y], axis=1).isna().sum()
summary.round(2)

## Correlation between design variables and functional attributes

Pearson correlation on pairwise-complete rows. With ~30 samples, |r| below roughly 0.35
is not distinguishable from zero at the 5 % level, so read the weak cells with care.

In [ ]:
corr = pd.concat([ds.X, ds.Y], axis=1).corr().loc[FEATURES, TARGETS]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(TARGETS)), TARGETS, rotation=40, ha="right")
ax.set_yticks(range(len(FEATURES)), FEATURES)
for i in range(len(FEATURES)):
    for j in range(len(TARGETS)):
        ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("Design variables vs functional attributes")
fig.tight_layout()

## Single-variable baselines

A simple reference in the spirit of de Weck (2006): for each target, an ordinary
least-squares fit on the single most correlated design variable. These are in-sample
fits, so they are an upper bound for such a simple model, not a validated score.
`e_RMSE` is the RMSE as a percentage of the target mean.

In [ ]:
from sklearn.linear_model import LinearRegression

X_filled = ds.X.fillna(ds.X.mean())
fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.6))
rows = []
for ax, target in zip(axes, TARGETS):
    best = corr[target].abs().idxmax()
    mask = ds.Y[target].notna()
    x, y = X_filled.loc[mask, [best]], ds.Y.loc[mask, target]
    fit = LinearRegression().fit(x, y)
    pred = fit.predict(x)
    r = np.corrcoef(x[best], y)[0, 1]
    e_rmse = 100 * np.sqrt(np.mean((pred - y) ** 2)) / y.mean()
    rows.append({"target": target, "best single variable": best, "r": r, "R2": r**2, "e_RMSE_%": e_rmse})
    ax.scatter(x[best], y, s=18)
    order = np.argsort(x[best].values)
    ax.plot(x[best].values[order], pred[order], color="black")
    ax.set_xlabel(best)
    ax.set_ylabel(target)
    ax.set_title(f"R² = {r**2:.2f}")
fig.tight_layout()
pd.DataFrame(rows).set_index("target").round(3)

## Outliers

Rows with a |z-score| above 2.5 in any column. On small datasets a single unusual
vehicle can dominate a fold, so these are worth checking before modelling.

In [ ]:
values = pd.concat([ds.X, ds.Y], axis=1)
z = (values - values.mean()) / values.std()
outliers = z.where(z.abs() > 2.5).stack().dropna().rename("z").to_frame()
outliers["value"] = [values.at[idx, col] for idx, col in outliers.index]
outliers.rename_axis(["vehicle", "column"]).round(2)